In [ ]:
from swarmbots.mj_env.scenarios.obstacle_street_scenario import ObstacleStreetScenario
from swarmbots.mj_env.swarm.homogeneous_swarm import HomogeneousSwarm
from swarmbots.mj_env.swarm.unit_config import UNIT_CONFIG_TETRAHEDRON_ZX
from swarmbots.mj_env.swarm_bots_env import SwarmBotsEnv
%load_ext autoreload
%autoreload 2

import mujoco
from rendering import display_video

import numpy as np

In [ ]:

# swarm = SimpleSwarmCubeZX(connection_torquescale=10)
swarm = HomogeneousSwarm(
    unit_start_locations=[
        (0, 0, 0),
        (-0.5, 0, 0),
        (0.5, 0, 0),
    ],
    unit_config=UNIT_CONFIG_TETRAHEDRON_ZX,
    connection_torquescale=10
)
scenario = ObstacleStreetScenario(
    swarm, 
    payload_type='sphere', 
    payload_size=(0.2, 0.05, 0.25),
    payload_mass=5,
    payload_start_location_offset=(0, 0, 1), 
    include_connectors_xpos_in_obs=True,
    include_connectors_xquat_in_obs=True,
    seed=None
)

opt = mujoco.MjvOption()

env = SwarmBotsEnv(
    scenario=scenario,
    render_mode="rgb_array",
    width=640,
    height=480,
    scene_option=opt,
    action_repeat=15,
    episode_length=50,
    camera=0
)

rng = np.random.default_rng()


frames = []
for i in range(10):
    done = False
    obs, info = env.reset()

    while not done:
        # Random action
        action = env.action_space.sample()
        action['actuators'][:] = 0
        action['connectors'][:] = False
        obs, reward, terminated, truncated, info = env.step(action)

        frame = env.render()
        if frame is not None:
            frames.append(frame)

        done = terminated or truncated

        if info and 'error' in info:
            print('err ' + str(env.data.time))


        # if len(frames) % 50 == 0:
        #     mj_env.data.eq_active[:] = 0
        #     mj_env.data.eq_active[rng.integers(low=0, high=len(mj_env.data.eq_active))] = 1

    print(f"Recorded {len(frames)} frames")
    for _ in range(15):
        frames.append(np.zeros_like(frames[0]))

# mj_env.close()
display_video(frames, 30)

In [ ]:
env.observation_space

In [ ]:
env.action_space

In [ ]:
env.model.body('Unit0--main_body')

In [ ]:
env.scenario.get_obs(
    model=env.model,
    data=env.data,
    state=env.scenario_state,
    connections=env.swarm_connections,
)

In [ ]:
is_active, _, _ = env.swarm_connections.get_active_connections()
is_active

In [ ]:
action = np.asarray(env.action_space.sample()['connectors'], dtype=bool)
action

In [ ]:
np.stack(np.where(np.logical_and(action, np.logical_not(is_active)))).T

In [ ]:
np.where(np.logical_and(is_active, np.logical_not(action)))

In [ ]:
type(env.data.warning[mujoco.mjtWarning.mjWARN_BADQACC].lastinfo)

In [ ]:
env.model.eq_data[:5]

In [ ]:
env.scenario.data.qpos.shape

In [ ]:
[b.name for b in env.scenario.spec.bodies]

In [ ]:
env.scenario.spec.body('Unit0--main_body')

In [ ]:
import torch
from torch.distributions import Categorical

dist = Categorical(torch.tensor([0.6, 0.3, 0.1]))
torch.exp(dist.log_prob(torch.tensor([3.0])))